# Fitting RRF / Blend Weights from Labeled Data

When you combine multiple ranked lists (e.g. dense vector search + BM25 keyword search),
the default is equal weights `[1.0, 1.0]`. But one retriever may be systematically better
for your domain. `RRFWeightFitter` learns the best weights from a small set of labeled queries.

**No AWS required** — this notebook uses synthetic data to show the full workflow.

## 1. Imports

In [ ]:
from dynavec import RRFWeightFitter
from dynavec.eval import LabeledQuery
from dynavec.models import SearchResult

## 2. Create synthetic retriever results

Imagine two retrievers for a banking FAQ search engine:
- **Dense** (vector/semantic): good at finding conceptually similar docs
- **Sparse** (BM25/keyword): good at exact term matches but noisier

We simulate 4 queries. Dense consistently ranks the relevant doc higher;
sparse often buries it.

In [ ]:
def r(doc_id, score):
    return SearchResult(id=doc_id, score=score)

# Each list = results for one query from that retriever
# Shape: [retriever][query_index] -> list[SearchResult]

dense_results = [
    # query 0: "FD interest rate" — dense puts doc_fd first ✓
    [r("doc_fd", 0.95), r("doc_savings", 0.60), r("doc_noise", 0.20)],
    # query 1: "how to open savings account" — dense puts doc_savings first ✓
    [r("doc_savings", 0.90), r("doc_loan", 0.50), r("doc_noise", 0.15)],
    # query 2: "home loan EMI calculator" — dense puts doc_loan first ✓
    [r("doc_loan", 0.88), r("doc_fd", 0.40), r("doc_noise", 0.10)],
    # query 3: "UPI transaction limit" — both retrievers agree ✓
    [r("doc_upi", 0.92), r("doc_fd", 0.45), r("doc_savings", 0.20)],
]

sparse_results = [
    # query 0: sparse buries doc_fd at rank 2 — noisy keyword match ✗
    [r("doc_savings", 0.80), r("doc_noise", 0.50), r("doc_fd", 0.20)],
    # query 1: sparse buries doc_savings at rank 2 ✗
    [r("doc_loan", 0.75), r("doc_noise", 0.45), r("doc_savings", 0.15)],
    # query 2: sparse buries doc_loan at rank 2 ✗
    [r("doc_fd", 0.70), r("doc_noise", 0.40), r("doc_loan", 0.10)],
    # query 3: sparse also gets this right ✓
    [r("doc_upi", 0.85), r("doc_fd", 0.40), r("doc_savings", 0.20)],
]

print(f"Retrievers: 2")
print(f"Queries: {len(dense_results)}")

## 3. Define labeled (ground-truth) queries

`LabeledQuery` holds the query text and the set of relevant document IDs.
These are your human-judged relevance labels.

In [ ]:
labeled_queries = [
    LabeledQuery(query="FD interest rate",            relevant_ids={"doc_fd"}),
    LabeledQuery(query="how to open savings account", relevant_ids={"doc_savings"}),
    LabeledQuery(query="home loan EMI calculator",    relevant_ids={"doc_loan"}),
    LabeledQuery(query="UPI transaction limit",       relevant_ids={"doc_upi"}),
]

print("Labeled queries:")
for lq in labeled_queries:
    print(f"  '{lq.query}' → relevant: {lq.relevant_ids}")

## 4. Build the fitter and check baseline score

First, see how well equal weights `[1.0, 1.0]` perform.
This is the default RRF behavior — no fitting.

In [ ]:
# result_lists shape: [retriever][query] -> list[SearchResult]
fitter = RRFWeightFitter(
    labeled_queries=labeled_queries,
    result_lists=[dense_results, sparse_results],
    k=60,       # RRF smoothing constant (standard default)
    eval_k=3,   # evaluate nDCG@3
)

baseline = fitter.score([1.0, 1.0])
all_dense = fitter.score([1.0, 0.0])
all_sparse = fitter.score([0.0, 1.0])

print(f"nDCG@3 — equal weights  [1.0, 1.0]: {baseline:.4f}")
print(f"nDCG@3 — dense only     [1.0, 0.0]: {all_dense:.4f}")
print(f"nDCG@3 — sparse only    [0.0, 1.0]: {all_sparse:.4f}")

## 5. Fit weights with grid search

Grid search tries `w1 = 0.00, 0.01, 0.02, ..., 1.00` and `w2 = 1 - w1`.
It returns the combination with the highest mean nDCG@3 on your labeled data.

In [ ]:
best_weights = fitter.fit(method="grid", n_points=100)
fitted_score = fitter.score(best_weights)

print(f"Best weights:  dense={best_weights[0]:.2f}, sparse={best_weights[1]:.2f}")
print(f"nDCG@3 after fitting: {fitted_score:.4f}")
print(f"Improvement over equal weights: +{fitted_score - baseline:.4f}")

## 6. Sweep weights to visualize the score landscape

This shows exactly why the fitter picks the weights it does.

In [ ]:
import numpy as np

w1_values = np.linspace(0, 1, 101)
scores = [fitter.score([w1, 1.0 - w1]) for w1 in w1_values]

try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(8, 4))
    plt.plot(w1_values, scores, color="steelblue", linewidth=2)
    plt.axvline(best_weights[0], color="tomato", linestyle="--", label=f"best w_dense={best_weights[0]:.2f}")
    plt.axhline(baseline, color="gray", linestyle=":", label=f"equal weights baseline ({baseline:.3f})")
    plt.xlabel("w_dense  (w_sparse = 1 - w_dense)")
    plt.ylabel("mean nDCG@3")
    plt.title("nDCG@3 vs. RRF weight on dense retriever")
    plt.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    # Print a text table if matplotlib is not installed
    print(f"{'w_dense':>8}  {'w_sparse':>8}  {'nDCG@3':>8}")
    for w1, s in zip(w1_values[::10], scores[::10]):
        marker = " ← best" if abs(w1 - best_weights[0]) < 0.01 else ""
        print(f"{w1:8.2f}  {1-w1:8.2f}  {s:8.4f}{marker}")

## 7. Three-retriever example with random search

When you have 3+ retrievers, grid search doesn't scale (3D grid = 10,000+ points).
Use `method="random"` instead — it samples random points from the weight simplex.

In [ ]:
# Third retriever: re-ranking model (good but slow, used as a third signal)
rerank_results = [
    [r("doc_fd", 0.99), r("doc_savings", 0.50), r("doc_loan", 0.20)],
    [r("doc_savings", 0.97), r("doc_fd", 0.45), r("doc_loan", 0.15)],
    [r("doc_loan", 0.96), r("doc_savings", 0.30), r("doc_fd", 0.10)],
    [r("doc_upi", 0.98), r("doc_savings", 0.35), r("doc_fd", 0.10)],
]

fitter3 = RRFWeightFitter(
    labeled_queries=labeled_queries,
    result_lists=[dense_results, sparse_results, rerank_results],
    eval_k=3,
)

best3 = fitter3.fit(method="random", n_points=500)
score3 = fitter3.score(best3)

print(f"Best weights (3 retrievers):")
print(f"  dense={best3[0]:.3f}, sparse={best3[1]:.3f}, rerank={best3[2]:.3f}")
print(f"nDCG@3: {score3:.4f}")

## 8. Use the fitted weights in production

Pass the learned weights directly to `reciprocal_rank_fusion`:

In [ ]:
from dynavec import reciprocal_rank_fusion

# In production, call each retriever, then fuse with learned weights
new_dense_results  = [r("doc_fd", 0.91), r("doc_savings", 0.55)]
new_sparse_results = [r("doc_savings", 0.70), r("doc_fd", 0.35)]

fused = reciprocal_rank_fusion(
    [new_dense_results, new_sparse_results],
    weights=best_weights,   # ← the weights RRFWeightFitter learned
)

print("Fused ranking with fitted weights:")
for rank, res in enumerate(fused, 1):
    print(f"  {rank}. {res.id}  (RRF score={res.score:.6f})")